# cell2location — Step 2 aggregation: combine chunked outputs

Combine per-chunk results from `step2_spatial_mapping.ipynb` runs into a
single AnnData. Only needed if `n_chunks > 1`; otherwise skip.

The aggregation pattern follows the cell2state_embryo workflow:
- load the original full spatial AnnData;
- copy each chunk's `obsm` keys back at the chunk's indices;
- optionally save chunk models under `uns['mod_batch{i}']`.

See [issue #356](https://github.com/BayraktarLab/cell2location/issues/356) and
[issue #375](https://github.com/BayraktarLab/cell2location/issues/375) for the
rationale.


In [ ]:
# === PARAMETERS (papermill) ===
spatial_h5ad_path = ""              # original full spatial AnnData (BEFORE chunking)
chunk_h5ad_glob = ""                # glob pattern, e.g. "./output/c2l_run_chunk*/sp.h5ad"
n_chunks = 1
output_path = "./spatial_mapping_output/full_aggregated.h5ad"
obsm_keys = ["means", "q05", "q50", "q95"]
include_chunk_models = False        # if True, attach each chunk's mod to uns (large!)


In [ ]:
import os
import glob as _glob
import numpy as np
import scanpy as sc


## Aggregate chunk outputs

In [ ]:
adata_full = sc.read_h5ad(spatial_h5ad_path)
print(f"Full AnnData: {adata_full.n_obs} locations, {adata_full.n_vars} genes")

chunk_paths = sorted(_glob.glob(chunk_h5ad_glob))
if len(chunk_paths) != n_chunks:
    raise ValueError(f"Found {len(chunk_paths)} chunks at {chunk_h5ad_glob!r}, expected {n_chunks}")

# Initialise obsm slots
for i, chunk_path in enumerate(chunk_paths):
    adata_chunk = sc.read_h5ad(chunk_path)
    # Discover number of cell types from the first chunk's first key
    if i == 0:
        first_key = obsm_keys[0]
        n_cell_types = adata_chunk.obsm[first_key].shape[1]
        for k in obsm_keys:
            adata_full.obsm[k] = np.zeros((adata_full.n_obs, n_cell_types), dtype='float32')

    # Map chunk indices to full-adata indices
    chunk_idx = adata_full.obs.index.get_indexer(adata_chunk.obs.index)
    if (chunk_idx < 0).any():
        raise ValueError(f"Chunk {i} has locations not present in the full AnnData. "
                         f"Did the chunk h5ad come from a different parent dataset?")
    for k in obsm_keys:
        if k in adata_chunk.obsm:
            adata_full.obsm[k][chunk_idx] = adata_chunk.obsm[k]

    if include_chunk_models and 'mod' in adata_chunk.uns:
        adata_full.uns[f'mod_batch{i}'] = adata_chunk.uns['mod']
    print(f"Aggregated chunk {i}: {len(chunk_idx)} locations")

os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
adata_full.write(output_path)
print(f"Wrote aggregated output: {output_path}")
